# How can we anticipate workplace insurance claims?

**MATH4SDG / UDSM — optional, student-led backup project**

**The problem:** An imaginary insurer covers establishments of different sizes, sectors and operating histories. How well can a simple model estimate **the number of claims in a future year**? What additional evidence would be needed before using such a model for decisions?

> **DATA ARE ENTIRELY SYNTHETIC.** No numbers, employers, programme effects or rates in this project come from WCF, TIRA, an insurance company or any actual Tanzanian institution. The generated 'costs' are arbitrary classroom units, **not TZS**. Do not present results as facts about any real sector or country.

All records are produced locally and reproducibly; **no download, credentials or private industry data are required**. The notebook is intended for group work, with optional extensions rather than a fixed teaching timetable. See [project guide](../projects/insurance-claims/README.md).

Code is MIT licensed, educational explanations CC BY 4.0, and the artificial dataset CC0 1.0.

## 1. Define a useful prediction problem

Suppose one establishment covers 20 workers for a full year and another covers 300. Even if their accident risk is identical, which establishment would you expect to report more claims?

We call **worker-years** the total **exposure**: 50 workers covered for half a year contribute 25 worker-years. The **claims frequency** is

$$\hat\lambda = \frac{\text{number of claims}}{\text{worker-years of exposure}}.$$

We first evaluate a model using the simulated outcomes for **2025** (as if they were not yet known), then use the evaluated approach to forecast **expected claims for 2026**, whose outcomes are not supplied. A forecast is an expectation, not an exact prediction. What information would an actual insurer or WCF have at the **start** of a year? Which columns would only be known afterwards?

## Load the example data

We begin with a few simple Python operations. The small `bootstrap` helper connects notebooks to the workshop's existing Python tools; you do not need to understand Python import paths to work on the investigation. Keep this notebook directly in the `notebooks/` folder for now.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_poisson_deviance
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

import bootstrap
from tools.synthetic_insurance import generate, SYNTHETIC_MARKER
from tools.insurance_methods import (
    aggregate_rate, poisson_rate_interval, split_by_time, validate,
)

In [ ]:
data = generate()  # All rows are fictional.
validate(data)
print("SYNTHETIC TRAINING DATA ONLY — NOT REAL INSURANCE STATISTICS")

First inspect the shape of the table (rows, columns):

In [ ]:
data.shape

Which years do the artificial records cover?

In [ ]:
sorted(data["year"].unique().tolist())

View the first five records:

In [ ]:
data.head()

## 2. Inspect the data rather than trusting a ready-made table

Each row is a *synthetic establishment-year*. The variables `worker_years`, `claims_count`, `sector`, `zone` and `programme` are fictional. `total_claim_cost_units` is determined **after** claims and must never enter a model pretending to predict the year's claims in advance.

Questions: Are there missing sector labels? Are exposures positive? Is the number of claims an integer? Why might repeated years for the same establishment make the observations dependent?

In [ ]:
data.isna().sum()

Look at the distribution of fictional claim counts. The `ax` variable represents the plot, and we set its label directly:

In [ ]:
ax = data["claims_count"].plot.hist(
    bins=35,
    figsize=(8, 3),
    title="Invented claims per establishment-year",
)
ax.set_xlabel("Synthetic claim count")

## 3. Compare exposure-adjusted claim rates by sector

**Do not average individual establishment claim frequencies without weighting.** A small establishment with one claim can have an extremely high observed frequency. Aggregate by dividing the **total number of claims** by the **total exposure**.

Does a high total claim count necessarily indicate a high *rate*? Can you think of factors that might explain an observed association, besides the sector itself?

First make a copy so that replacing missing sector labels does not change our original dataset:

In [ ]:
clean_data = data.copy()
clean_data["sector"] = clean_data["sector"].fillna("Unreported")

Next add up claim counts and worker-years for each sector. The `groupby()` operation combines rows with the same sector:

In [ ]:
by_sector = clean_data.groupby("sector")[["claims_count", "worker_years"]].sum()
by_sector

Calculate claims per 100 worker-years. This divides the **sum of claims** by the **sum of exposure** for each sector, rather than averaging individual establishment rates:

In [ ]:
by_sector["claims_per_100_worker_years"] = (
    100 * by_sector["claims_count"] / by_sector["worker_years"]
)
by_sector.round(2)

Plot the rates using the returned Matplotlib `Axes` object:

In [ ]:
ax = by_sector["claims_per_100_worker_years"].sort_values().plot.barh(
    figsize=(9, 4),
    title="SYNTHETIC claims frequency (not real sector statistics)",
)
ax.set_xlabel("Claims per 100 worker-years")
ax.set_ylabel("Synthetic sector")

## 4. Evaluation must respect time

Imagine asking, at the beginning of 2025, how many claims will arrive that year. Using actual 2025 claims or costs as predictors would be **data leakage**. A random split of all years can obscure whether the model works in a new year.

We train on **2021–2024** and keep **2025** untouched until final evaluation. The same imaginary establishment can occur in several years. For a truly *new-establishment* prediction problem, a different split and evaluation would be needed.

**2025 is for checking the model, not our final forecasting target.** We will subsequently forecast 2026 using assumed future exposure and predictors.

In [ ]:
train, test = split_by_time(data, test_year=2025)
sorted(train["year"].unique().tolist())

In [ ]:
sorted(test["year"].unique().tolist())

## 5. A baseline model you can understand from scratch

If every establishment has the same rate, our simplest predictor is

$$\hat\lambda_{\mathrm{base}} = \frac{\sum_{i\in\mathrm{train}} N_i}{\sum_{i\in\mathrm{train}} e_i},\qquad
\widehat{N}_i = e_i\hat\lambda_{\mathrm{base}},$$

where $e_i$ is worker-year exposure and $N_i$ is observed claims. Why do we multiply the predicted **rate** by exposure when predicting a claim **count**?

We can evaluate the predictions using **Poisson deviance**, a measure of discrepancy between the observed count $N$ and the positive predicted mean $\mu$:

$d(N,\mu)=2\left(N\log\frac{N}{\mu}-N+\mu\right).$

When $N=0$ we take $N\log(N/\mu)=0$. The deviance is zero for a perfect prediction and larger for worse predictions. Scikit-learn's `mean_poisson_deviance` averages this quantity over the observations. We compare our models on the **same held-out 2025 rows**, giving each establishment-year equal weight. The function requires strictly positive predicted counts.

In [ ]:
baseline_rate = aggregate_rate(train)
print(f"Historical baseline: {100 * baseline_rate:.3f} claims per 100 worker-years")

Calculate how many 2025 claims the baseline expects, then evaluate its predictions against the synthetic 2025 outcomes:

In [ ]:
baseline_predicted = baseline_rate * test["worker_years"].to_numpy()
y_test = test["claims_count"].to_numpy()
base_score = mean_poisson_deviance(y_test, baseline_predicted)
round(base_score, 3)

## 6. A Poisson regression model

One standard count model is

$$N_i\mid x_i,e_i \sim \operatorname{Poisson}(e_i\lambda_i),\qquad
\log(\lambda_i)=\beta_0 + \beta^\top x_i.$$

The *log of exposure* is an **offset** in the count mean, while $x_i$ represents predictors known before the outcome. We use scikit-learn's `PoissonRegressor` to predict **counts per worker-year**, weighting rows by exposure. This corresponds to the count model's exposure adjustment; the size and interpretation of a regularization penalty also depend on the objective's normalization. [Official scikit-learn insurance example](https://scikit-learn.org/stable/auto_examples/linear_model/plot_tweedie_regression_insurance_claims.html).

The predictors below are `sector`, `zone` and `programme`, and **never** the observed claim count or total paid amount. Missing sectors are explicitly labelled "Unreported". What assumptions does this model make? Why might a fitted association with a training programme be **non-causal**?

**Step 1 — Choose the predictors.** A regression model needs numerical inputs. Our three predictors are categories, and we explicitly mark missing sector values as `"Unreported"`. We do not include the year's observed claims or costs, because these would leak the answer.

In [ ]:
features = ["sector", "zone", "programme"]
X_train = train[features].fillna("Unreported")
X_train.head()

**Step 2 — Define what we predict.** We fit the **claims rate per worker-year**, with the exposure used as a weight. Why would a rate observed over 300 worker-years usually be more informative than a rate observed over 3 worker-years?

In [ ]:
rate_train = train["claims_count"] / train["worker_years"]
exposure_train = train["worker_years"]
rate_train.head()

**Step 3 — Convert categories to numbers.** `OneHotEncoder` turns categories such as `"Transport"` and `"Retail"` into indicator columns. `ColumnTransformer` applies that conversion to the selected predictors. Ask AI to show what the first three encoded rows look like, then verify using the code.

In [ ]:
encoder = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown="ignore"), features)
])
encoder

**Step 4 — Assemble a model.** The `Pipeline` first encodes categories and then fits Poisson regression. `alpha=0.02` controls regularization (a penalty discouraging large coefficients); it is an arbitrary modelling choice, not a known constant. More regularization does not automatically mean a better model.

In [ ]:
model = Pipeline([
    ("features", encoder),
    ("poisson", PoissonRegressor(alpha=0.02, max_iter=1000))
])
model

**Step 5 — Fit on historical data only.** Notice how the weight is passed to the `poisson` step inside the pipeline: `poisson__sample_weight`. Nothing from the observed 2025 claim counts is used here.

In [ ]:
model.fit(
    X_train,
    rate_train,
    poisson__sample_weight=exposure_train.to_numpy()
)

**Step 6 — Predict the 2025 claim rate, then recover expected counts.** The model predicts a rate. We multiply by each establishment's worker-years to estimate its expected number of claims.

In [ ]:
X_test = test[features].fillna("Unreported")
predicted_rate = model.predict(X_test)
model_predicted = predicted_rate * test["worker_years"].to_numpy()

pd.Series(model_predicted[:5], name="First five expected claim counts").round(2)

**Step 7 — Score the fitted model.** Compare the predicted counts with the actual synthetic 2025 counts using the same metric as the baseline.

In [ ]:
model_score = mean_poisson_deviance(y_test, model_predicted)
round(model_score, 3)

Compare the predictions on the **same held-out 2025 records**. A smaller Poisson deviance is better:

In [ ]:
pd.DataFrame({
    "model": ["Constant rate baseline", "Poisson regression"],
    "mean_Poisson_deviance_2025": [base_score, model_score],
    "expected_2025_claims": [baseline_predicted.sum(), model_predicted.sum()],
    "actual_2025_claims": [y_test.sum(), y_test.sum()],
}).round(2)

### Optional investigation — should we weight the evaluation by exposure?

So far we have averaged the **count** deviance equally across establishment-years. We can instead calculate the deviance of the **claim rates**, using worker-years as weights:

```python
mean_poisson_deviance(
    observed_count / exposure,
    predicted_count / exposure,
    sample_weight=exposure,
)
```

**Mathematical challenge:** Show that $d(N,e\lambda)=e\,d(N/e,\lambda)$ for Poisson deviance. Consequently, on the same test set, the exposure-weighted mean deviance of rates equals the unweighted *sum of count deviances* divided by total exposure. It differs from the mean count deviance by the constant factor $n/\sum_i e_i$.

Therefore **both scores rank models identically on the same evaluation rows**, although their numerical units and scales differ. Compare the two numerical values, and explain why the ranking does not change.

In [ ]:
exposure_test = test["worker_years"].to_numpy()
rate_actual = y_test / exposure_test
rate_baseline = baseline_predicted / exposure_test
rate_model = model_predicted / exposure_test

pd.DataFrame({
    "model": ["Constant rate baseline", "Poisson regression"],
    "mean_count_deviance": [base_score, model_score],
    "mean_deviance_per_worker_year": [
        mean_poisson_deviance(rate_actual, rate_baseline, sample_weight=exposure_test),
        mean_poisson_deviance(rate_actual, rate_model, sample_weight=exposure_test),
    ],
}).round(4)

## 7. Check where the model works and fails

A model with a lower overall deviance can still systematically underpredict some sectors or a particular year. Compare the *observed* and *expected* rates in 2025 by sector. Why is an aggregate comparison easier to interpret than individual claim-count residuals?

**Warning:** A synthetic sector-level association is not evidence for a real sector-specific risk. Our sample has recurring establishments and unobserved differences in risk.

Prepare the data for grouping. The 2025 model predictions are *expected counts*, not observed counts:

In [ ]:
comparison = test[["sector", "worker_years", "claims_count"]].copy()
comparison["sector"] = comparison["sector"].fillna("Unreported")
comparison["baseline_count"] = baseline_predicted
comparison["model_count"] = model_predicted

Aggregate the 2025 outcomes and predictions by sector:

In [ ]:
groups = comparison.groupby("sector")[
    ["worker_years", "claims_count", "baseline_count", "model_count"]
].sum()
groups

Turn totals into rates per 100 worker-years:

In [ ]:
for name in ["claims_count", "baseline_count", "model_count"]:
    groups[name + "_per_100"] = 100 * groups[name] / groups["worker_years"]

groups[["claims_count_per_100", "baseline_count_per_100", "model_count_per_100"]].round(2)

Compare those rates visually, again by working directly with an `Axes` object:

In [ ]:
ax = groups[
    ["claims_count_per_100", "baseline_count_per_100", "model_count_per_100"]
].plot.bar(figsize=(10, 4), title="SYNTHETIC 2025: observed and predicted claim rates")
ax.set_ylabel("Claims per 100 worker-years")
ax.set_xlabel("Synthetic sector")
ax.tick_params(axis="x", rotation=25)

## 8. Forecast expected claims for 2026

We have evaluated the modelling method on **2025**. Now imagine it is November 2026 and we want an expectation for 2026, not a claim that we know the year's final outcomes.

The synthetic dataset has **no 2026 outcomes or exposure plans**. For illustration, make a transparent *planning scenario*: assume each fictional establishment will have the same worker-year exposure, sector, zone and programme status as in 2025. These are assumptions for a forecast, **not observations from 2026**.

We can now refit on all available 2021–2025 outcomes, because these outcomes belong to the historical dataset. This should not be used as an excuse to revise our model repeatedly based on the 2025 test results. Ask: Which assumed 2026 inputs might be most uncertain?

In [ ]:
scenario_2026 = test[
    ["establishment_id", "sector", "zone", "programme", "worker_years"]
].copy()
scenario_2026["year"] = 2026  # A hypothetical planning year.
scenario_2026.head()

Fit the already chosen models on the full 2021–2025 historical dataset and predict 2026 *expected claim counts*:

In [ ]:
from sklearn.base import clone

# Copy the model's configuration, not the fitted parameters.
final_model = clone(model)
X_full = data[features].fillna("Unreported")
rate_full = data["claims_count"] / data["worker_years"]

final_model.fit(
    X_full,
    rate_full,
    poisson__sample_weight=data["worker_years"].to_numpy(),
)

Apply the refitted model to the **assumed 2026 inputs**. These forecasts are expected claim counts, not observed 2026 outcomes.

In [ ]:
baseline_rate_2026 = aggregate_rate(data)
scenario_2026["baseline_expected_claims"] = (
    baseline_rate_2026 * scenario_2026["worker_years"]
)

X_2026 = scenario_2026[features].fillna("Unreported")
forecast_rate_2026 = final_model.predict(X_2026)
scenario_2026["model_expected_claims"] = (
    forecast_rate_2026 * scenario_2026["worker_years"].to_numpy()
)

Summarize the two forecasts. **There are no synthetic 2026 realized claim counts to compare with these forecasts.**

In [ ]:
pd.Series({
    "Assumed 2026 worker-years": scenario_2026["worker_years"].sum(),
    "Baseline expected 2026 claims": scenario_2026["baseline_expected_claims"].sum(),
    "Poisson model expected 2026 claims": scenario_2026["model_expected_claims"].sum(),
}).round(1)

## 9. Confidence in the rate — and limits of a neat interval

As a starting point, if $K$ independent events follow a Poisson distribution with mean $E\lambda$ for a known exposure $E$, an exact count-based confidence interval can be converted into a rate interval by dividing its endpoints by $E$.

But our imaginary establishments have **persistent unobserved risk differences**, so a simple pooled Poisson interval may understate uncertainty. The data are also repeated over years and conditions change in 2025. Statistical precision conditional on a model does not imply reliable real-world decisions.

In [ ]:
historical_claims = int(train["claims_count"].sum())
historical_exposure = float(train["worker_years"].sum())
historical_interval = poisson_rate_interval(historical_claims, historical_exposure)
pd.Series({
    "Historical claims per 100 worker-years": 100 * historical_claims / historical_exposure,
    "Poisson-only 95% lower bound": 100 * historical_interval[0],
    "Poisson-only 95% upper bound": 100 * historical_interval[1],
}).round(3)

Compute the same interval separately for the held-out 2025 outcomes. These intervals do **not** account for repeated establishments or changes in conditions:

In [ ]:
test_claims = int(test["claims_count"].sum())
test_exposure = float(test["worker_years"].sum())
test_interval = poisson_rate_interval(test_claims, test_exposure)
pd.Series({
    "2025 claims per 100 worker-years": 100 * test_claims / test_exposure,
    "Poisson-only 95% lower bound": 100 * test_interval[0],
    "Poisson-only 95% upper bound": 100 * test_interval[1],
}).round(3)

## 10. Open the model: why does the baseline work?

Under the simplified assumptions $N_i\sim\operatorname{Poisson}(e_i\lambda)$, the log-likelihood (apart from constants) is

$$\ell(\theta) = \sum_i [N_i\theta - e_i\exp(\theta)],\quad \theta=\log\lambda.$$

Its derivative is $\sum_i (N_i - e_i\exp(\theta))$. Verify with Python that this derivative is approximately zero for the baseline rate computed above.

**AI activity:** Ask an AI assistant to explain the derivative and provide a minimal implementation from scratch. Then independently check shapes, differentiation, the role of exposure, and whether the proposed code accidentally uses 2025 data.

In [ ]:
theta = np.log(baseline_rate)
expected_training = train["worker_years"].to_numpy() * np.exp(theta)
gradient = np.sum(train["claims_count"].to_numpy() - expected_training)
assert abs(gradient) < 1e-6
round(float(gradient), 9)

## 11. Optional extensions and group presentation

Choose **one** question rather than trying to implement every model:

- **Model checking:** Could an overdispersed or zero-inflated model describe the counts better? Check residuals; do not select models by test-year performance repeatedly.
- **Distribution shift:** Compare training and held-out claim rates. Is the poorer performance in 2025 due to a new risk environment, unobserved factors, or a modelling problem? You cannot determine causality from these columns alone.
- **Claim severity:** The synthetic `total_claim_cost_units` is a heavy-tailed outcome. Study **frequency versus severity** and why predicting total costs is a different problem. Costs are generated only after claims and must not be used as a predictor of those same claims.
- **Validation:** If you want to tune model complexity, tune using earlier years (e.g., 2021–2023 versus 2024) and reserve 2025 for final evaluation.
- **Fairness and practical deployment:** How would you check whether an establishment's apparent risk reflects reporting behavior, data collection practices, or other confounders? Avoid automated decisions affecting workers or establishments from this synthetic exercise.

**Five-minute workshop report:** Include the 2025 validation result and a clearly labelled 2026 planning-scenario forecast, with no claim to know actual 2026 outcomes.  (1) precise prediction question and units, (2) dataset provenance and its artificial status, (3) baseline and model comparison, (4) one limitation or uncertainty, (5) one question for an industry representative.

**Never upload confidential partner data to this public repository or share them with third-party AI systems without authorization.**